# Bangalore House Price Prediction — Retraining Notebook

This notebook reproduces the original project's data cleaning and feature
engineering pipeline, **removes a data-leakage bug found in the original
notebook**, compares several model architectures on the corrected feature
set, and exports the final production artifacts used by the Streamlit app.

## Background: the leakage bug

The original notebook computed `log_price_per_sqft` — a value derived
directly from `price` (`price × 100000 / total_sqft`, capped, then
log-transformed) — for outlier-detection purposes. However, it was
accidentally left in the model's input features (`X`), so the model was
partially "predicting" the price using a feature computed from the price
itself. This inflated the original test R² to **0.973**, which was not a
genuine measure of predictive power: a real user cannot supply
`log_price_per_sqft` before knowing the price they're trying to predict.

This notebook fixes that by dropping `price_per_sqft` and
`log_price_per_sqft` entirely from the model's feature set, and reports the
honest (lower, but real) performance that results.


## 1. Load raw data

In [1]:
import pandas as pd

df = pd.read_csv("BHP.csv")  # place BHP.csv in the same folder as this notebook
print(df.shape)
df.head()


(13320, 9)


,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00


## 2. Cleaning

Identical to the original notebook's cleaning steps:
- Drop `society` (41% missing)
- Impute `balcony` (mean), `bath` (median), `location` (rare-category
  fallback), `size` (default "2BHK")


In [2]:
df = df.drop(["society"], axis=1)

df["balcony"] = df["balcony"].fillna(df["balcony"].mean().round())
df["balcony"] = df["balcony"].round().astype(int)

location_counts = df["location"].value_counts()
rare_locations = location_counts[location_counts < 10].index
df["location"] = df["location"].apply(lambda x: "other" if x in rare_locations else x)
df["location"] = df["location"].fillna("Whitefield")

df["size"] = df["size"].fillna("2BHK")
df["bath"] = df["bath"].fillna(df["bath"].dropna().median())

df.isna().sum()


area_type       0
availability    0
location        0
size            0
total_sqft      0
bath            0
balcony         0
price           0
dtype: int64

## 3. Feature engineering

- Extract `bhk` from the `size` text field
- Convert `total_sqft` range strings (e.g. `"2100 - 2850"`) to their average
- Remove rows with an unrealistic `total_sqft`/`bhk` ratio (< 300 sqft per
  bedroom) — matches the original notebook's outlier-removal rule
- Cap `bhk` at 8


In [3]:
df["bhk"] = df["size"].str.extract(r"(\d+)").astype(float).astype("Int64")


def ConvertRange(x):
    try:
        if "-" in str(x):
            temp = x.split("-")
            if len(temp) == 2:
                return (float(temp[0]) + float(temp[1])) / 2
        return float(x)
    except Exception:
        return None


df["total_sqft"] = df["total_sqft"].apply(ConvertRange)

# price_per_sqft is computed ONLY to decide which rows are outliers below.
# It is never kept as a model input -- this is exactly where the original
# notebook's leakage came from, so we're deliberate about dropping it after use.
df["price_per_sqft"] = df["price"] * 100000 / df["total_sqft"]

df = df[(df["total_sqft"] / df["bhk"]) >= 300]
df["price_per_sqft"] = df["price_per_sqft"].clip(lower=500, upper=25000)
df["bhk"] = df["bhk"].apply(lambda x: 8 if x > 8 else x).astype(int)

print("Rows after cleaning/outlier removal:", df.shape[0])


Rows after cleaning/outlier removal: 12530


## 4. Drop leaked / unused columns

`size` and `area_type` are superseded by extracted features. `price_per_sqft`
(and its log-transform, which the original notebook mistakenly kept) are
dropped here -- this is the fix for the leakage bug described above.


In [4]:
df = df.drop(columns=["size", "area_type", "price_per_sqft"])
print("Final columns:", df.columns.tolist())


Final columns: ['availability', 'location', 'total_sqft', 'bath', 'balcony', 'price', 'bhk']


## 5. Encode + train/test split

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

categorical_cols = ["availability", "location"]
preprocessor = ColumnTransformer(
    transformers=[("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)],
    remainder="passthrough",
)

X = df.drop(columns=["price"])
y = df["price"]
print("Feature columns (X):", X.columns.tolist())

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


Feature columns (X): ['availability', 'location', 'total_sqft', 'bath', 'balcony', 'bhk']


## 6. Compare model architectures

Random Forest (baseline + tuned), Gradient Boosting, and XGBoost are compared
on the identical, leakage-free feature set.


In [6]:
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor


def evaluate(name, model):
    pipe = Pipeline(steps=[("preprocessor", preprocessor), ("regressor", model)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2_test = r2_score(y_test, y_pred)
    r2_train = pipe.score(X_train, y_train)
    print(f"{name:35s} RMSE={rmse:8.3f}  MAE={mae:7.3f}  R2_test={r2_test:6.4f}  R2_train={r2_train:6.4f}")
    return pipe


_ = evaluate("Random Forest (baseline, n=100)", RandomForestRegressor(n_estimators=100, random_state=42))
_ = evaluate(
    "Random Forest (tuned)",
    RandomForestRegressor(n_estimators=400, max_depth=14, min_samples_leaf=2, max_features=0.6, random_state=42, n_jobs=-1),
)
_ = evaluate(
    "Gradient Boosting",
    GradientBoostingRegressor(n_estimators=400, learning_rate=0.05, max_depth=3, random_state=42),
)
_ = evaluate(
    "XGBoost",
    XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=5, subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0, random_state=42, n_jobs=-1),
)


Random Forest (baseline, n=100)     RMSE=  91.070  MAE= 31.706  R2_test=0.6119  R2_train=0.9271


Random Forest (tuned)               RMSE=  89.977  MAE= 33.754  R2_test=0.6212  R2_train=0.8038


Gradient Boosting                   RMSE=  87.303  MAE= 34.338  R2_test=0.6434  R2_train=0.7944


XGBoost                             RMSE=  94.819  MAE= 33.757  R2_test=0.5793  R2_train=0.8740


**Result:** Gradient Boosting had the best test R²/RMSE and the smallest
train/test gap (least overfitting), so it was selected as the final model.


## 7. Train the final model

In [7]:
model_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", GradientBoostingRegressor(
        n_estimators=400, learning_rate=0.05, max_depth=3, random_state=42
    )),
])

model_pipeline.fit(X_train, y_train)

y_pred = model_pipeline.predict(X_test)
rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
mae = float(mean_absolute_error(y_test, y_pred))
r2_test = float(r2_score(y_test, y_pred))
r2_train = float(model_pipeline.score(X_train, y_train))

print(f"Final Gradient Boosting -- RMSE={rmse:.3f} MAE={mae:.3f} R2_test={r2_test:.4f} R2_train={r2_train:.4f}")


Final Gradient Boosting -- RMSE=87.303 MAE=34.338 R2_test=0.6434 R2_train=0.7944


## 8. Export artifacts for the Streamlit app

Saves everything `utils/prediction.py` needs: the trained pipeline, the
exact feature order, the valid category lists (for building dropdowns), and
the evaluation metrics (for the "About the Model" section).


In [8]:
import os
import json
import joblib

os.makedirs("model_out", exist_ok=True)
joblib.dump(model_pipeline, "model_out/Bangalore-project.joblib")

feature_columns = X.columns.tolist()
with open("model_out/feature_columns.json", "w") as f:
    json.dump(feature_columns, f)

ohe = model_pipeline.named_steps["preprocessor"].named_transformers_["cat"]
categories = {col: sorted(cats.tolist()) for col, cats in zip(categorical_cols, ohe.categories_)}
with open("model_out/categories.json", "w") as f:
    json.dump(categories, f, indent=2)

metrics = {
    "model_type": "GradientBoostingRegressor",
    "hyperparameters": {"n_estimators": 400, "learning_rate": 0.05, "max_depth": 3, "random_state": 42},
    "rmse": rmse,
    "mae": mae,
    "r2_test": r2_test,
    "r2_train": r2_train,
    "n_rows_final": int(df.shape[0]),
    "n_features": len(feature_columns),
    "feature_columns": feature_columns,
    "bhk_range": [int(df["bhk"].min()), int(df["bhk"].max())],
    "total_sqft_range": [float(df["total_sqft"].min()), float(df["total_sqft"].max())],
    "bath_range": [int(df["bath"].min()), int(df["bath"].max())],
    "balcony_range": [int(df["balcony"].min()), int(df["balcony"].max())],
    "n_availability_categories": len(categories["availability"]),
    "n_location_categories": len(categories["location"]),
}
with open("model_out/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Saved to model_out/ -- copy these 4 files into the app's model/ folder.")


Saved to model_out/ -- copy these 4 files into the app's model/ folder.


## Summary

| Metric | Original (leaked) | Retrained (honest) |
|---|---|---|
| R² (test) | 0.973 | **0.643** |
| MAE | 2.53 Lakhs | 34.34 Lakhs |
| RMSE | 23.81 Lakhs | 87.30 Lakhs |

The corrected model is the one actually shipped in this project's
`model/` folder and used by the Streamlit app.
